## 0. Paths and Config

In [ ]:
# Set SNT Paths
SNT_ROOT_PATH  <- "~/workspace"
CODE_PATH      <- file.path(SNT_ROOT_PATH, "code")
CONFIG_PATH    <- file.path(SNT_ROOT_PATH, "configuration")
PIPELINE_PATH  <- file.path(SNT_ROOT_PATH, "pipelines", "snt_dhis2_population_transformation")
REPORTING_NB_PATH <- file.path(PIPELINE_PATH, "reporting")

In [ ]:
# load util functions
source(file.path(CODE_PATH, "snt_utils.r"))
source(file.path(CODE_PATH, "snt_palettes.r"))
source(file.path(PIPELINE_PATH, "utils", "snt_dhis2_population_transformation_report.r"))

# Execute function
install_and_load(c("tidyverse", "arrow", "sf", "reticulate", "glue"))

# Set environment to load openhexa.sdk from the right environment
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")
Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")

# Load openhexa.sdk
reticulate::py_config()$python
openhexa <- import("openhexa.sdk")

### 0.1 Load configuration

In [ ]:
# Load SNT config
config_json <- tryCatch({ jsonlite::fromJSON(file.path(CONFIG_PATH, "SNT_config.json"))},
    error = function(e) {
        msg <- paste0("Error while loading configuration", conditionMessage(e))  
        cat(msg)   
        stop(msg) 
    }) 

# Configuration variables
dataset_id <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_POPULATION_TRANSFORMATION
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
COUNTRY_NAME <- config_json$SNT_CONFIG$COUNTRY_NAME
ADM_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)

## 1. Import data

In [ ]:
# Transformed population data from dataset
population_data <- tryCatch({ get_latest_dataset_file_in_memory(dataset_id, paste0(COUNTRY_CODE, "_population.parquet")) }, 
                  error = function(e) {
                      msg <- paste0(COUNTRY_NAME , " Population data is not available in dataset : " , dataset_id, " last version.")
                      log_msg(msg, "warning")
                      population_data <- NULL
                      })

printdim(population_data)

In [ ]:
# shapes data from formatting dataset
shapes_dataset_name <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
shapes_data <- tryCatch({ get_latest_dataset_file_in_memory(shapes_dataset_name, paste0(COUNTRY_CODE, "_shapes.geojson")) }, 
                  error = function(e) {                      
                      msg <- paste0(COUNTRY_NAME , " Shapes data is not available in dataset : " , shapes_dataset_name, " last version.")
                      log_msg(msg, "warning")
                      shapes_data <- NULL
                      })

printdim(shapes_data)

In [ ]:
# Pipeline parameters
pop_transform_ds_id <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_POPULATION_TRANSFORMATION
parameters_data <- tryCatch({ get_latest_dataset_file_in_memory(pop_transform_ds_id, paste0(COUNTRY_CODE, "_parameters.json")) }, 
                  error = function(e) {                      
                      msg <- paste0(COUNTRY_NAME , " parameters data is not available in dataset : " , pop_transform_ds_id, " last version.")
                      log_msg(msg, "warning")
                      shapes_data <- NULL
                      })

## 2. Carte des populations désagrégée par niveau administrative 2  

> Compute plot breaks

In [ ]:
# Metadata columns
metadata_columns <- c('YEAR', 'ADM1_NAME', 'ADM1_ID', 'ADM2_NAME', 'ADM2_ID')
indicators <- setdiff(colnames(population_data), metadata_columns)

# Select the years to plot
# year_reference <- parameters_data$YEAR_REFERENCE
# years_to_keep <- c(year_reference-1, year_reference, year_reference+1)
# population_data_filtered <- population_data %>% dplyr::filter(YEAR %in% years_to_keep)

n_breaks <- 5

# Compute labels for each indicator
label_breaks_list <- list()
for (indicator in indicators) {
    pop_values <- population_data[[indicator]]  # compute for all the years, these are projected pop values anyway
    value_breaks <- compute_value_intervals(pop_values, n_breaks, style="kmeans")
    labels <- create_dynamic_labels(value_breaks)
    label_breaks_list[[indicator]] <- list(
            value_breaks = value_breaks,
            labels = labels
        )
}


> Generate plots

In [ ]:
# Plot 
palette_population <- c(
    "1" = "#fae6db",
    "2" = "#f1b195",
    "3" = "#ea7354",
    "4" = "#cc3f32",
    "5" = "#972620",
    "6" = "#5e0f0f" 
)

for (indicator in indicators) {
    col_selection <- c(metadata_columns, indicator)
    
    plot <- build_population_choropleth(
      population_data_filtered = population_data %>% dplyr::select(all_of(col_selection)),
      shapes_data = shapes_data,
      population_column = indicator,
      breaks_values = label_breaks_list[[indicator]]$value_breaks,
      labels = label_breaks_list[[indicator]]$labels,
      legend_title = glue("{indicator}:"),
      plot_title = glue("{indicator} par niveau administratif 2"),
      palette_values = palette_population
    )
    
    # Export to see better in high resolution
    output_file = file.path(REPORTING_NB_PATH, "outputs", "figures", glue("{COUNTRY_CODE}_choropleth_poptransformed_{indicator}.png"))
    ggsave(
      filename = output_file,
      create.dir = TRUE,
      units = "cm",
      width = 21,
      height = 15,
      dpi = 300
    )
    log_msg(glue("Population figure saved: {output_file}"))
}

In [ ]:
for (indicator in indicators) {
    IRdisplay::display_png(file = file.path(REPORTING_NB_PATH, "outputs", "figures", glue("{COUNTRY_CODE}_choropleth_poptransformed_{indicator}.png")))
}